In [42]:
from langchain_openai.llms import OpenAI
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from dotenv import load_dotenv

load_dotenv()

llm = OpenAI(model="gpt-3.5-turbo-instruct")

chat = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [17]:
prompt_template = PromptTemplate.from_template("""
    Responda a seguinte pergunta do usuário:
    {question}
""")

print(prompt_template.format(question="Oque é um Saas"))


    Responda a seguinte pergunta do usuário:
    Oque é um Saas



In [18]:
prompt_template = PromptTemplate.from_template("""
    Responda a seguinte pergunta do usuário em até {n_words} palavras:
    {question}
""")

print(prompt_template.format(question="Oque é um Saas", n_words=20))


    Responda a seguinte pergunta do usuário em até 20 palavras:
    Oque é um Saas



In [19]:
prompt_template = PromptTemplate.from_template("""
    Responda a seguinte pergunta do usuário em até {n_words} palavras:
    {question}
""", partial_variables={"n_words": 5})

print(prompt_template.format(question="Oque é um Saas"))


    Responda a seguinte pergunta do usuário em até 5 palavras:
    Oque é um Saas



# Utilizando Múltiplos Prompts

In [21]:
template_word_count = PromptTemplate.from_template("""
    Responda a seguinte pergunta do usuário em até {n_words} palavras.
""")

template_line_count = PromptTemplate.from_template("""
    Responda a seguinte pergunta do usuário em até {n_lines} linhas.
""")

template_language = PromptTemplate.from_template("""
    Retorne a resposta em {language}.
""")

final_template = (template_word_count + template_line_count + template_language +
                  "Responsa a pergunta seguindo as instruções {question}")

print(final_template)

input_variables=['language', 'n_lines', 'n_words', 'question'] input_types={} partial_variables={} template='\n    Responda a seguinte pergunta do usuário em até {n_words} palavras.\n\n    Responda a seguinte pergunta do usuário em até {n_lines} linhas.\n\n    Retorne a resposta em {language}.\nResponsa a pergunta seguindo as instruções {question}'


In [23]:
final_prompt = final_template.format(n_words=15, n_lines=4, language="English", question="Oque é o sol")

llm.invoke(final_prompt)

'\n\n\nO sol é uma estrela que fornece luz e calor para a Terra. Ele está localizado no centro do nosso sistema solar.'

In [24]:
print(final_prompt)


    Responda a seguinte pergunta do usuário em até 15 palavras.

    Responda a seguinte pergunta do usuário em até 4 linhas.

    Retorne a resposta em English.
Responsa a pergunta seguindo as instruções Oque é o sol


## Templates para Chat

In [26]:
chat_template = ChatPromptTemplate.from_template("Essa é minha dúvida: {duvida}")
chat_template.format_messages(duvida="Quem é você?")

[HumanMessage(content='Essa é minha dúvida: Quem é você?', additional_kwargs={}, response_metadata={})]

In [28]:
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "Você é um assistente irônico e se chama {nome_assistente}"),
        ("human", "Olá como vai?"),
        ("ai", "Estou bem, como posso lhe ajudar?"),
        ("human", "{pergunta}")
    ]
)

chat_template

ChatPromptTemplate(input_variables=['nome_assistente', 'pergunta'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['nome_assistente'], input_types={}, partial_variables={}, template='Você é um assistente irônico e se chama {nome_assistente}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Olá como vai?'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Estou bem, como posso lhe ajudar?'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pergunta'], input_types={}, partial_variables={}, template='{pergunta}'), additional_kwargs={})])

In [29]:
chat_template.format_messages(nome_assistente="Bot Sos", pergunta="Qual seu nome?")

[SystemMessage(content='Você é um assistente irônico e se chama Bot Sos', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Olá como vai?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Estou bem, como posso lhe ajudar?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Qual seu nome?', additional_kwargs={}, response_metadata={})]

In [32]:
chat.invoke(chat_template.format_messages(nome_assistente="Bot Sos", pergunta="Qual seu nome?"))

AIMessage(content='Meu nome é Bot Sos, seu assistente irônico de hoje. Como posso ser útil?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 55, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVImpcLKQmDBYHdH6PIQCfJxyeoQl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d96df-fa32-75e3-b649-4fc85084a32a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 24, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Few Shot Prompt

In [36]:
examples = [
    {"pergunta": "Qual é a maior montanha do mundo, o Monte Everest ou o K2?",
     "resposta":
     """São necessárias perguntas de acompanhamento aqui: Sim.
Pergunta de acompanhamento: Qual é a altura do Monte Everest?
Resposta intermediária: O Monte Everest tem 8.848 metros de altura.
Pergunta de acompanhamento: Qual é a altura do K2?
Resposta intermediária: O K2 tem 8.611 metros de altura.
Então a resposta final é: Monte Everest
""",
    },
    {"pergunta": "Quem nasceu primeiro, Charles Darwin ou Albert Einstein?",
     "resposta":
     """São necessárias perguntas de acompanhamento aqui: Sim.
Pergunta de acompanhamento: Quando nasceu Charles Darwin?
Resposta intermediária: Charles Darwin nasceu em 12 de fevereiro de 1809.
Pergunta de acompanhamento: Quando nasceu Albert Einstein?
Resposta intermediária: Albert Einstein nasceu em 14 de março de 1879.
Então a resposta final é: Charles Darwin
""",
    },
    {"pergunta": "Quem foi o pai de Napoleão Bonaparte?",
     "resposta":
     """São necessárias perguntas de acompanhamento aqui: Sim.
Pergunta de acompanhamento: Quem foi Napoleão Bonaparte?
Resposta intermediária: Napoleão Bonaparte foi um líder militar e imperador francês.
Pergunta de acompanhamento: Quem foi o pai de Napoleão Bonaparte?
Resposta intermediária: O pai de Napoleão Bonaparte foi Carlo Buonaparte.
Então a resposta final é: Carlo Buonaparte
""",
    },
    {"pergunta": "Os filmes 'O Senhor dos Anéis' e 'O Hobbit' foram dirigidos pelo mesmo diretor?",
     "resposta":
     """São necessárias perguntas de acompanhamento aqui: Sim.
Pergunta de acompanhamento: Quem dirigiu 'O Senhor dos Anéis'?
Resposta intermediária: 'O Senhor dos Anéis' foi dirigido por Peter Jackson.
Pergunta de acompanhamento: Quem dirigiu 'O Hobbit'?
Resposta intermediária: 'O Hobbit' também foi dirigido por Peter Jackson.
Então a resposta final é: Sim
""",
    },
]

In [37]:
example_prompt = PromptTemplate(
    input_variables=["pergunta", "resposta"],
    template="Pergunta {pergunta}\n{resposta}"
)

example_prompt.format(**examples[0])

'Pergunta Qual é a maior montanha do mundo, o Monte Everest ou o K2?\nSão necessárias perguntas de acompanhamento aqui: Sim. \nPergunta de acompanhamento: Qual é a altura do Monte Everest? \nResposta intermediária: O Monte Everest tem 8.848 metros de altura. \nPergunta de acompanhamento: Qual é a altura do K2? \nResposta intermediária: O K2 tem 8.611 metros de altura. \nEntão a resposta final é: Monte Everest \n'

In [38]:
prompt = FewShotPromptTemplate(examples=examples,
                               example_prompt=example_prompt,
                               suffix="Pergunta: {input}",
                               input_variables=["input"])

In [39]:
print(prompt.format(input="Quem é melhor, Messi ou Cristiano Ronaldo"))

Pergunta Qual é a maior montanha do mundo, o Monte Everest ou o K2?
São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Qual é a altura do Monte Everest? 
Resposta intermediária: O Monte Everest tem 8.848 metros de altura. 
Pergunta de acompanhamento: Qual é a altura do K2? 
Resposta intermediária: O K2 tem 8.611 metros de altura. 
Então a resposta final é: Monte Everest 


Pergunta Quem nasceu primeiro, Charles Darwin ou Albert Einstein?
São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quando nasceu Charles Darwin? 
Resposta intermediária: Charles Darwin nasceu em 12 de fevereiro de 1809. 
Pergunta de acompanhamento: Quando nasceu Albert Einstein? 
Resposta intermediária: Albert Einstein nasceu em 14 de março de 1879. 
Então a resposta final é: Charles Darwin 


Pergunta Quem foi o pai de Napoleão Bonaparte?
São necessárias perguntas de acompanhamento aqui: Sim. 
Pergunta de acompanhamento: Quem foi Napoleão Bonaparte?

In [40]:
llm.invoke(prompt.format(input="Quem fez mais gols, Messi ou Cristiano Ronaldo?"))

'\nSão necessárias perguntas de acompanhamento aqui: Sim. \nPergunta de acompanhamento: Quantos gols Messi fez? \nResposta intermediária: Messi tem mais de 700 gols em sua carreira. \nPergunta de acompanhamento: Quantos gols Cristiano Ronaldo fez? \nResposta intermediária: Cristiano Ronaldo tem mais de 750 gols em sua carreira. \nEntão a resposta final é: Cristiano Ronaldo'

## Few Shot Prompting em Chat

In [41]:
example_prompt = ChatPromptTemplate.from_messages(
    [("human", "{pergunta}"),
     ("ai", "{resposta}")]
)

print(example_prompt.format_messages(**exemplos[0]))

[HumanMessage(content='Qual é a maior montanha do mundo, o Monte Everest ou o K2?', additional_kwargs={}, response_metadata={}), AIMessage(content='São necessárias perguntas de acompanhamento aqui: Sim. \nPergunta de acompanhamento: Qual é a altura do Monte Everest? \nResposta intermediária: O Monte Everest tem 8.848 metros de altura. \nPergunta de acompanhamento: Qual é a altura do K2? \nResposta intermediária: O K2 tem 8.611 metros de altura. \nEntão a resposta final é: Monte Everest \n', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [43]:
few_shot_template = FewShotChatMessagePromptTemplate(
    examples=exemplos,
    example_prompt=example_prompt
)

prompt_final = ChatPromptTemplate.from_messages([
    few_shot_template,
    ("human", "{input}")
])

prompt = prompt_final.format_messages(input="Quem fez mais gols, Messi ou Cristiano Ronaldo?")
# few_shot_template.format_messages()

In [44]:
chat.invoke(prompt)

AIMessage(content='São necessárias perguntas de acompanhamento aqui: Sim.\nPergunta de acompanhamento: Quantos gols Messi marcou em sua carreira?\nResposta intermediária: Até o momento, Lionel Messi marcou mais de 700 gols em sua carreira.\nPergunta de acompanhamento: Quantos gols Cristiano Ronaldo marcou em sua carreira?\nResposta intermediária: Cristiano Ronaldo também marcou mais de 700 gols em sua carreira.\nEntão, é necessário verificar os números mais atualizados para determinar quem fez mais gols até o momento.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 520, 'total_tokens': 643, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVIuMdqehpRqJID4ME5